In [ ]:
!pip install -q fair-esm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 10.1 MB/s eta 0:00:00


In [ ]:
import pathlib
import torch
import pandas as pd

from esm import FastaBatchedDataset, pretrained


In [ ]:
# Set up file system
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:

def extract_embeddings(model_name, fasta_file, output_dir, tokens_per_batch=4096, seq_length=1022,repr_layers=[33]):

    model, alphabet = pretrained.load_model_and_alphabet(model_name)
    model.eval()

    if torch.cuda.is_available():
        model = model.cuda()

    dataset = FastaBatchedDataset.from_file(fasta_file)
    batches = dataset.get_batch_indices(tokens_per_batch, extra_toks_per_seq=1)

    data_loader = torch.utils.data.DataLoader(
        dataset,
        collate_fn=alphabet.get_batch_converter(seq_length),
        batch_sampler=batches
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    full_output = {}

    with torch.no_grad():
        for batch_idx, (labels, strs, toks) in enumerate(data_loader):

            print(f'Processing batch {batch_idx + 1} of {len(batches)}')

            if torch.cuda.is_available():
                toks = toks.to(device="cuda", non_blocking=True)

            out = model(toks, repr_layers=repr_layers, return_contacts=False)

            logits = out["logits"].to(device="cpu")
            representations = {layer: t.to(device="cpu") for layer, t in out["representations"].items()}

            for i, label in enumerate(labels):
                entry_id = label.split()[0]

                filename = output_dir / f"{entry_id}.pt"
                truncate_len = min(seq_length, len(strs[i]))

                result = {"entry_id": entry_id}
                # TODO do I want this representation? Yes, I think so (because it is consistent regardless of length)
                result["mean_representations"] = {
                        layer: t[i, 1 : truncate_len + 1].mean(0).clone()
                        for layer, t in representations.items()
                    }

                if len(repr_layers) == 1:
                    full_output[entry_id] = representations[repr_layers[0]][i, 1 : truncate_len + 1].mean(0).clone()

                # torch.save(result, filename)

    # output full_output as a tsv (with first column as the entry id and the rest of the values in other columns)
    full_filename = output_dir / "all_embeddings.tsv"

    pd.DataFrame.from_dict(full_output).T.to_csv(full_filename, sep='\t')

In [ ]:

model_name = 'esm2_t33_650M_UR50D'
fasta_file = pathlib.Path('/content/drive/MyDrive/Courses/STATS305c/all_peptides.from_features.UPPER.fasta')
output_dir = pathlib.Path('/content/drive/MyDrive/Courses/STATS305c/embeddings/')

extract_embeddings(model_name, fasta_file, output_dir)

Processing batch 1 of 89
Processing batch 2 of 89
Processing batch 3 of 89
Processing batch 4 of 89
Processing batch 5 of 89
Processing batch 6 of 89
Processing batch 7 of 89
Processing batch 8 of 89
Processing batch 9 of 89
Processing batch 10 of 89
Processing batch 11 of 89
Processing batch 12 of 89
Processing batch 13 of 89
Processing batch 14 of 89
Processing batch 15 of 89
Processing batch 16 of 89
Processing batch 17 of 89
Processing batch 18 of 89
Processing batch 19 of 89
Processing batch 20 of 89
Processing batch 21 of 89
Processing batch 22 of 89
Processing batch 23 of 89
Processing batch 24 of 89
Processing batch 25 of 89
Processing batch 26 of 89
Processing batch 27 of 89
Processing batch 28 of 89
Processing batch 29 of 89
Processing batch 30 of 89
Processing batch 31 of 89
Processing batch 32 of 89
Processing batch 33 of 89
Processing batch 34 of 89
Processing batch 35 of 89
Processing batch 36 of 89
Processing batch 37 of 89
Processing batch 38 of 89
Processing batch 39 o

In [ ]:
embeddings_df = pd.read_csv('/content/drive/MyDrive/Courses/STATS305c/embeddings/all_embeddings.tsv', sep='\t')
embeddings_df.set_index('Unnamed: 0', inplace=True)
embeddings_df.index.name = 'peptide_id'


In [ ]:
embeddings_df

,0,1,2,3,4,5,6,7,8,9,...,1270,1271,1272,1273,1274,1275,1276,1277,1278,1279
peptide_id,,,,,,,,,,,,,,,,,,,,,
958,0.153072,0.125646,0.199581,0.022823,-0.038598,0.009050,-0.140163,0.395465,0.313469,-0.035303,...,0.110433,-0.067498,-0.061101,0.289990,-0.078648,0.156445,0.188686,0.244827,-0.131866,-0.169149
7914,-0.037616,0.024300,0.011779,0.003787,-0.058849,-0.083176,-0.147703,0.073233,0.167967,-0.084098,...,-0.061601,-0.076131,-0.152620,0.082052,0.110218,0.117759,0.067548,0.124962,-0.042451,0.025906
7915,-0.037616,0.024300,0.011779,0.003787,-0.058849,-0.083176,-0.147703,0.073233,0.167967,-0.084098,...,-0.061601,-0.076131,-0.152620,0.082052,0.110218,0.117759,0.067548,0.124962,-0.042451,0.025906
7916,-0.037616,0.024300,0.011779,0.003787,-0.058849,-0.083176,-0.147703,0.073233,0.167967,-0.084098,...,-0.061601,-0.076131,-0.152620,0.082052,0.110218,0.117759,0.067548,0.124962,-0.042451,0.025906
7917,-0.037616,0.024300,0.011779,0.003787,-0.058849,-0.083176,-0.147703,0.073233,0.167967,-0.084098,...,-0.061601,-0.076131,-0.152620,0.082052,0.110218,0.117759,0.067548,0.124962,-0.042451,0.025906
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3112,-0.094340,-0.230119,-0.021702,-0.036959,0.067585,-0.148612,0.103549,-0.145244,0.013519,-0.067115,...,-0.132314,-0.138110,-0.088714,0.003145,0.060051,-0.034476,-0.123539,0.038579,0.180379,-0.030784
6552,-0.000343,-0.080601,-0.108853,0.056486,-0.120614,-0.019980,0.086382,-0.028744,-0.093015,0.147370,...,-0.007572,0.009964,0.084187,0.070503,0.082919,0.014564,0.245692,-0.102310,-0.031039,0.027673
11343,0.061899,-0.086030,0.029204,-0.060958,0.082928,-0.073561,0.066286,-0.103133,-0.067862,-0.041841,...,0.160319,-0.083233,-0.171989,-0.023415,0.104689,-0.096890,0.024865,-0.007521,-0.004038,0.002466
